In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DATA_DIR = Path("/content/drive/MyDrive/CSCI 5527 Project/Dataset Preprocessing/processed_eeg_dataset/processed_eeg_dataset")

X_eeg = np.load(DATA_DIR / "X_eeg.npy", mmap_mode="r")
groups = np.load(DATA_DIR / "groups.npy", allow_pickle=True)

metadata_sss = pd.read_csv(DATA_DIR / "eeg_metadata_with_sss.csv")
metadata_kss = pd.read_csv(DATA_DIR / "eeg_metadata_with_kss.csv")

print("X_eeg shape:", X_eeg.shape)
print("groups shape:", groups.shape)
print("metadata_sss shape:", metadata_sss.shape)
print("metadata_kss shape:", metadata_kss.shape)

X_eeg shape: (8300, 61, 1250)
groups shape: (8300,)
metadata_sss shape: (8300, 14)
metadata_kss shape: (8300, 14)


In [5]:
TARGET_NAME = "SSS"

if TARGET_NAME == "SSS":
    metadata = metadata_sss.copy()
elif TARGET_NAME == "KSS":
    metadata = metadata_kss.copy()
else:
    raise ValueError("TARGET_NAME must be 'SSS' or 'KSS'")

print("Using target:", TARGET_NAME)
print(metadata.columns.tolist())

Using target: SSS
['subject_id', 'session', 'task', 'label', 'label_name', 'epoch_index', 'sfreq', 'n_channels', 'n_times', 'img_channels', 'img_height', 'img_width', 'file_path', 'SSS']


In [6]:
metadata[TARGET_NAME] = pd.to_numeric(metadata[TARGET_NAME], errors="coerce")

print("Missing target rows:", metadata[TARGET_NAME].isna().sum())

valid_mask = metadata[TARGET_NAME].notna().to_numpy()
valid_indices = np.where(valid_mask)[0]

metadata_valid = metadata.iloc[valid_indices].reset_index(drop=True)
groups_valid = groups[valid_indices]

print("Valid labeled epochs:", len(metadata_valid))
print("Valid subjects:", metadata_valid["subject_id"].nunique())

Missing target rows: 3920
Valid labeled epochs: 4380
Valid subjects: 40


In [7]:
target_values = sorted(metadata_valid[TARGET_NAME].dropna().unique().tolist())
value_to_rank = {v: i for i, v in enumerate(target_values)}
rank_to_value = {i: v for v, i in value_to_rank.items()}
num_classes = len(target_values)

print("Observed target values:", target_values)
print("Num ordinal classes:", num_classes)
print("Mapping:", value_to_rank)

Observed target values: [1.0, 2.0, 3.0, 4.0, 5.0, 7.0]
Num ordinal classes: 6
Mapping: {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3, 5.0: 4, 7.0: 5}


In [8]:
metadata_valid["target_rank"] = metadata_valid[TARGET_NAME].map(value_to_rank)
print(metadata_valid[[TARGET_NAME, "target_rank"]].drop_duplicates().sort_values(TARGET_NAME))

      SSS  target_rank
60    1.0            0
180   2.0            1
0     3.0            2
480   4.0            3
660   5.0            4
1860  7.0            5


In [9]:
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

all_positions = np.arange(len(metadata_valid))
dummy_y = metadata_valid["target_rank"].to_numpy()

gss_1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=SEED)
train_pos, temp_pos = next(gss_1.split(all_positions, dummy_y, groups_valid))

temp_y = dummy_y[temp_pos]
temp_groups = groups_valid[temp_pos]
relative_val = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

gss_2 = GroupShuffleSplit(n_splits=1, train_size=relative_val, random_state=SEED + 1)
val_pos_local, test_pos_local = next(gss_2.split(np.arange(len(temp_pos)), temp_y, temp_groups))

val_pos = temp_pos[val_pos_local]
test_pos = temp_pos[test_pos_local]

print("train:", len(train_pos))
print("val  :", len(val_pos))
print("test :", len(test_pos))

train: 3060
val  : 720
test : 600


In [10]:
train_groups = set(groups_valid[train_pos].astype(str))
val_groups = set(groups_valid[val_pos].astype(str))
test_groups = set(groups_valid[test_pos].astype(str))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0

print("No subject leakage.")

No subject leakage.


In [11]:
def ordinal_encode(rank, num_classes):
    levels = torch.zeros(num_classes - 1, dtype=torch.float32)
    levels[:rank] = 1.0
    return levels

In [12]:
class EEGOrdinalDataset(Dataset):
    def __init__(self, x_eeg, metadata_df, valid_indices, split_positions, num_classes):
        self.x_eeg = x_eeg
        self.metadata_df = metadata_df.reset_index(drop=True)
        self.valid_indices = np.asarray(valid_indices, dtype=np.int64)
        self.split_positions = np.asarray(split_positions, dtype=np.int64)
        self.num_classes = num_classes

    def __len__(self):
        return len(self.split_positions)

    def __getitem__(self, idx):
        pos = int(self.split_positions[idx])
        global_idx = int(self.valid_indices[pos])

        x = np.array(self.x_eeg[global_idx], dtype=np.float32, copy=True)
        x = torch.from_numpy(x).unsqueeze(0)   # (1, 61, 1250)

        row = self.metadata_df.iloc[pos]
        rank = int(row["target_rank"])
        levels = ordinal_encode(rank, self.num_classes)

        meta = {
            "global_index": global_idx,
            "subject_id": row["subject_id"],
            "session": row["session"],
            "file_path": row["file_path"],
            "epoch_index": int(row["epoch_index"]),
            TARGET_NAME: float(row[TARGET_NAME]),
        }

        return x, levels, rank, meta

In [13]:
train_dataset = EEGOrdinalDataset(X_eeg, metadata_valid, valid_indices, train_pos, num_classes)
val_dataset = EEGOrdinalDataset(X_eeg, metadata_valid, valid_indices, val_pos, num_classes)
test_dataset = EEGOrdinalDataset(X_eeg, metadata_valid, valid_indices, test_pos, num_classes)

train_ranks = metadata_valid.iloc[train_pos]["target_rank"].to_numpy(dtype=np.int64)
class_counts = np.bincount(train_ranks, minlength=num_classes)
sample_weights = 1.0 / np.maximum(class_counts[train_ranks], 1)

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Class counts:", class_counts)

Class counts: [300 840 480 480 840 120]


In [14]:
class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, act="relu"):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

        if act == "relu":
            self.act = nn.ReLU(inplace=True)
        elif act == "leaky_relu":
            self.act = nn.LeakyReLU(0.1, inplace=True)
        elif act == "silu":
            self.act = nn.SiLU(inplace=True)
        else:
            raise ValueError(f"Unsupported activation: {act}")

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

In [15]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        pooled = self.pool(x).view(b, c)
        scale = self.fc(pooled).view(b, c, 1, 1)
        return x * scale

In [16]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, dropout=0.10):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(channels, channels, kernel_size=3, padding=1, act="silu"),
            nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))

In [17]:
def initialize_custom_model(module):
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
            if getattr(m, "weight", None) is not None:
                nn.init.ones_(m.weight)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

In [18]:
class ResidualEEGCNNOrdinal(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBNAct(1, 32, kernel_size=(7, 31), padding=(3, 15), act="leaky_relu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
            ConvBNAct(32, 64, kernel_size=(5, 15), padding=(2, 7), act="silu"),
            nn.MaxPool2d(kernel_size=(2, 4)),
        )

        self.res_stack = nn.Sequential(
            ResidualBlock(64, dropout=0.10),
            SEBlock(64),
            ConvBNAct(64, 128, kernel_size=3, padding=1, act="silu"),
            nn.MaxPool2d(kernel_size=(2, 2)),
            ResidualBlock(128, dropout=0.15),
            SEBlock(128),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.55),
            nn.Linear(256, num_classes - 1),
        )

        initialize_custom_model(self)

    def forward(self, x):
        x = self.stem(x)
        x = self.res_stack(x)
        logits = self.head(x)
        return logits

In [19]:
model = ResidualEEGCNNOrdinal(num_classes=num_classes).to(DEVICE)
print(model)

ResidualEEGCNNOrdinal(
  (stem): Sequential(
    (0): ConvBNAct(
      (conv): Conv2d(1, 32, kernel_size=(7, 31), stride=(1, 1), padding=(3, 15), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): LeakyReLU(negative_slope=0.1, inplace=True)
    )
    (1): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
    (2): ConvBNAct(
      (conv): Conv2d(32, 64, kernel_size=(5, 15), stride=(1, 1), padding=(2, 7), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (3): MaxPool2d(kernel_size=(2, 4), stride=(2, 4), padding=0, dilation=1, ceil_mode=False)
  )
  (res_stack): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): ConvBNAct(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.

In [20]:
criterion = nn.BCEWithLogitsLoss()

In [21]:
def decode_ordinal_logits(logits):
    probs = torch.sigmoid(logits)
    pred_rank = (probs > 0.5).sum(dim=1)
    return pred_rank

In [22]:
def target_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearsonr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
        "Spearman": spearmanr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 else np.nan,
    }

In [23]:
def evaluate_ordinal_model(model, loader, device):
    model.eval()

    all_true_values = []
    all_pred_values = []
    running_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for x, levels, ranks, meta in loader:
            x = x.to(device)
            levels = levels.to(device)

            logits = model(x)
            loss = criterion(logits, levels)

            pred_rank = decode_ordinal_logits(logits).cpu().numpy()

            true_values = meta[TARGET_NAME]

            for i in range(len(pred_rank)):
                all_true_values.append(float(true_values[i]))
                all_pred_values.append(float(rank_to_value[int(pred_rank[i])]))

            running_loss += loss.item()
            n_batches += 1

    metrics = target_metrics(all_true_values, all_pred_values)

    return {
        "loss": running_loss / max(n_batches, 1),
        "metrics": metrics,
        "y_true": np.array(all_true_values, dtype=np.float32),
        "y_pred": np.array(all_pred_values, dtype=np.float32),
    }

In [24]:
optimizer = AdamW(model.parameters(), lr=2e-4, weight_decay=2e-3)

NUM_EPOCHS = 35
PATIENCE = 8

best_val_metric = -np.inf
best_state = None
stale_epochs = 0
history = []

In [25]:
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False)

    for x, levels, ranks, meta in progress:
        x = x.to(DEVICE)
        levels = levels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, levels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    val_out = evaluate_ordinal_model(model, val_loader, DEVICE)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_out["loss"],
        "val_MAE": val_out["metrics"]["MAE"],
        "val_RMSE": val_out["metrics"]["RMSE"],
        "val_Pearson": val_out["metrics"]["Pearson"],
        "val_Spearman": val_out["metrics"]["Spearman"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_MAE={val_out['metrics']['MAE']:.4f} | "
        f"val_RMSE={val_out['metrics']['RMSE']:.4f} | "
        f"val_Pearson={val_out['metrics']['Pearson']:.4f} | "
        f"val_Spearman={val_out['metrics']['Spearman']:.4f}"
    )

    current_metric = -val_out["metrics"]["RMSE"] + 0.1 * val_out["metrics"]["Pearson"]

    if current_metric > best_val_metric:
        best_val_metric = current_metric
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print("Early stopping triggered.")
            break

Epoch 1/35:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.4972 | val_MAE=2.4597 | val_RMSE=2.9000 | val_Pearson=-0.1008 | val_Spearman=-0.0799


Epoch 2/35:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 02 | train_loss=0.3617 | val_MAE=2.8403 | val_RMSE=3.1726 | val_Pearson=-0.0850 | val_Spearman=-0.0600


Epoch 3/35:   0%|          | 0/96 [00:00<?, ?it/s]

...

Epoch 18 | train_loss=0.0833 | val_MAE=1.5764 | val_RMSE=2.0300 | val_Pearson=0.0445 | val_Spearman=-0.0050


Epoch 19/35:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 19 | train_loss=0.0680 | val_MAE=1.2458 | val_RMSE=1.6103 | val_Pearson=0.1769 | val_Spearman=0.1865


Epoch 20/35:   0%|          | 0/96 [00:00<?, ?it/s]

Epoch 20 | train_loss=0.0550 | val_MAE=1.1806 | val_RMSE=1.5129 | val_Pearson=0.1118 | val_Spearman=0.1475
Early stopping triggered.


In [26]:
model.load_state_dict(best_state)

val_out = evaluate_ordinal_model(model, val_loader, DEVICE)
test_out = evaluate_ordinal_model(model, test_loader, DEVICE)

print("\nValidation:", val_out["metrics"])
print("Test      :", test_out["metrics"])


Validation: {'MAE': 1.0944443941116333, 'RMSE': np.float64(1.390443554302846), 'Pearson': np.float32(0.049418163), 'Spearman': np.float64(0.01988788460263805)}
Test      : {'MAE': 1.15666663646698, 'RMSE': np.float64(1.34783779524751), 'Pearson': np.float32(0.3441955), 'Spearman': np.float64(0.3547410824212002)}
